# 日志分析工具

本 notebook 用于分析日志文件，包括：
1. 分析总用时以及每个模块（以函数记）用时
2. 分析所有最后输出为 `is_answerable=False` 的问题的路径和原因
3. 总结和分析所有 `logger.error` 的产生和原因

## 使用说明
- 修改 `LOG_FILE_PATH` 变量可以分析其他日志文件
- 每个分析模块都是独立的，可以单独运行
- 可以方便地添加新的分析模块


## 1. 日志解析模块

定义日志解析的基础类和函数

In [69]:
# 导入必要的库
import re
from datetime import datetime
from collections import defaultdict, Counter
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import json
import pandas as pd

# 设置 pandas 显示选项，确保显示完整的列名和内容
pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', None)  # 显示所有行
pd.set_option('display.width', None)  # 不限制显示宽度
pd.set_option('display.max_colwidth', None)  # 不截断列内容，显示完整文本

# 测试版本标识（用于文件名）
TEST_VER = "0202"

# 日志文件路径
LOG_FILE_PATH = Path(f"../storage/testlog/{TEST_VER}test.log")

class LogEntry:
    """日志条目类"""
    def __init__(self, line: str):
        self.raw_line = line.strip()
        self.timestamp: Optional[datetime] = None
        self.level: Optional[str] = None
        self.module: Optional[str] = None
        self.function: Optional[str] = None
        self.line_number: Optional[int] = None
        self.message: Optional[str] = None
        
        self._parse()
    
    def _parse(self):
        """解析日志行，支持新的JSON格式和旧的文本格式"""
        # 首先尝试解析为JSON格式（新格式）
        try:
            log_data = json.loads(self.raw_line)
            if isinstance(log_data, dict) and 'record' in log_data:
                # 新格式：从 record 中提取信息
                record = log_data.get('record', {})
                
                # 提取时间戳
                time_info = record.get('time', {})
                if isinstance(time_info, dict):
                    # 尝试从 repr 字段解析
                    time_repr = time_info.get('repr', '')
                    if time_repr:
                        try:
                            # 处理带时区的格式: 2025-12-17 10:13:56.833959+08:00
                            if '+' in time_repr or time_repr.endswith('Z'):
                                # 使用 datetime.fromisoformat 处理带时区的时间
                                dt = datetime.fromisoformat(time_repr.replace('Z', '+00:00'))
                                # 转换为本地时间（去掉时区信息）
                                if dt.tzinfo:
                                    dt = dt.astimezone().replace(tzinfo=None)
                                self.timestamp = dt
                            else:
                                self.timestamp = datetime.strptime(time_repr, '%Y-%m-%d %H:%M:%S.%f')
                        except Exception:
                            # 如果解析失败，尝试使用 timestamp 字段
                            timestamp = time_info.get('timestamp')
                            if timestamp:
                                self.timestamp = datetime.fromtimestamp(timestamp)
                
                # 提取级别
                level_info = record.get('level', {})
                if isinstance(level_info, dict):
                    self.level = level_info.get('name', '')
                else:
                    self.level = str(level_info) if level_info else None
                
                # 提取模块名（从 name 字段中提取）
                name = record.get('name', '')
                if name:
                    # name 字段就是完整的模块路径，直接使用
                    self.module = name
                
                # 提取函数名
                self.function = record.get('function', '')
                
                # 提取行号
                line_num = record.get('line')
                if line_num is not None:
                    try:
                        self.line_number = int(line_num)
                    except (ValueError, TypeError):
                        pass
                
                # 提取消息
                self.message = record.get('message', '')
                
                # 如果解析成功，直接返回
                if self.timestamp:
                    return
        except (json.JSONDecodeError, KeyError, ValueError):
            # 不是JSON格式，继续尝试旧格式解析
            pass
        
        # 旧格式解析: 2025-11-14 17:32:53.305 | INFO | src.module:function:198 - message
        pattern = r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3}) \| (\w+)\s+\| ([^:]+):([^:]+):(\d+) - (.+)'
        match = re.match(pattern, self.raw_line)
        
        if match:
            try:
                self.timestamp = datetime.strptime(match.group(1), '%Y-%m-%d %H:%M:%S.%f')
                self.level = match.group(2)
                self.module = match.group(3)
                self.function = match.group(4)
                self.line_number = int(match.group(5))
                self.message = match.group(6)
            except Exception as e:
                pass
        else:
            # 尝试新格式的文本部分（从 text 字段中解析）
            # 格式: 2025-12-17 10:13:56.833|INFO |src.main:registerRouters:159| PID:9236| TID:26820| 消息
            text_pattern = r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3})\|(\w+)\s*\|([^:]+):([^:]+):(\d+)\|.*?\|(.+)'
            text_match = re.match(text_pattern, self.raw_line)
            if text_match:
                try:
                    self.timestamp = datetime.strptime(text_match.group(1), '%Y-%m-%d %H:%M:%S.%f')
                    self.level = text_match.group(2)
                    self.module = text_match.group(3)
                    self.function = text_match.group(4)
                    self.line_number = int(text_match.group(5))
                    self.message = text_match.group(6).strip()
                except Exception as e:
                    pass
    
    def __repr__(self):
        return f"LogEntry({self.timestamp}, {self.level}, {self.module}.{self.function})"


class LogParser:
    """日志解析器"""
    def __init__(self, log_file_path: Path):
        self.log_file_path = log_file_path
        self.entries: List[LogEntry] = []
        
    def parse(self) -> List[LogEntry]:
        """解析日志文件"""
        print(f"正在解析日志文件: {self.log_file_path}")
        with open(self.log_file_path, 'r', encoding='utf-8') as f:
            for line in f:
                entry = LogEntry(line)
                if entry.timestamp:  # 只保留成功解析的条目
                    self.entries.append(entry)
        
        print(f"解析完成，共 {len(self.entries)} 条有效日志")
        return self.entries
    
    def get_total_duration(self) -> float:
        """获取总时长（秒）"""
        if not self.entries:
            return 0
        start_time = self.entries[0].timestamp
        end_time = self.entries[-1].timestamp
        duration = (end_time - start_time).total_seconds()
        return duration
    
    def get_entries_by_level(self, level: str) -> List[LogEntry]:
        """按日志级别筛选"""
        return [e for e in self.entries if e.level == level.upper()]
    
    def get_entries_by_function(self, function_name: str) -> List[LogEntry]:
        """按函数名筛选"""
        return [e for e in self.entries if e.function == function_name]


# 解析日志
parser = LogParser(LOG_FILE_PATH)
entries = parser.parse()

# 显示基本信息
if len(entries) > 0:
    print(f"日志时间范围: {entries[0].timestamp} 到 {entries[-1].timestamp}")
    print(f"总时长: {parser.get_total_duration():.2f} 秒 ({parser.get_total_duration()/3600:.2f} 小时)")
    print(f"总日志条数: {len(entries)}")
else:
    print("警告: 日志文件为空或没有有效日志条目")
    print(f"总日志条数: {len(entries)}")

正在解析日志文件: ..\storage\testlog\0202test.log
解析完成，共 6498 条有效日志
日志时间范围: 2026-02-02 18:29:42.367732 到 2026-02-02 19:18:58.556086
总时长: 2956.19 秒 (0.82 小时)
总日志条数: 6498


## 2. 时间分析模块

分析总用时以及每个模块（以函数记）用时

##因程序并行难以准确衡量，已暂时放弃##


In [70]:
class TimeAnalyzer:
    """时间分析器"""
    def __init__(self, entries: List[LogEntry]):
        self.entries = entries
    
    def analyze_function_timing(self) -> pd.DataFrame:
        """分析每个函数的执行时间"""
        # 按函数分组，计算每个函数的开始和结束时间
        function_times = defaultdict(list)
        
        for entry in self.entries:
            if entry.function:
                key = f"{entry.module}.{entry.function}"
                function_times[key].append(entry.timestamp)
        
        # 计算每个函数的总用时
        results = []
        for func_name, timestamps in function_times.items():
            if len(timestamps) > 1:
                # 计算从第一次出现到最后一次出现的时间差
                duration = (max(timestamps) - min(timestamps)).total_seconds()
                results.append({
                    'function': func_name,
                    'start_time': min(timestamps),
                    'end_time': max(timestamps),
                    'duration_seconds': duration,
                    'call_count': len(timestamps)
                })
        
        df = pd.DataFrame(results)
        df = df.sort_values('duration_seconds', ascending=False)
        return df
    
    def analyze_function_duration_by_pattern(self) -> pd.DataFrame:
        """通过日志中的开始/完成模式分析函数执行时间"""
        # 查找包含"开始"、"启动"、"完成"、"结束"等关键词的日志
        start_patterns = ['开始', '启动', 'start', 'begin']
        end_patterns = ['完成', '结束', 'finish', 'done', '完毕']
        
        function_durations = []
        active_functions = {}  # {function_key: start_time}
        
        for entry in self.entries:
            if not entry.function or not entry.message:
                continue
            
            func_key = f"{entry.module}.{entry.function}"
            message_lower = entry.message.lower()
            
            # 检查是否是开始
            if any(pattern in message_lower for pattern in start_patterns):
                active_functions[func_key] = entry.timestamp
            
            # 检查是否是结束
            elif any(pattern in message_lower for pattern in end_patterns):
                if func_key in active_functions:
                    start_time = active_functions[func_key]
                    duration = (entry.timestamp - start_time).total_seconds()
                    function_durations.append({
                        'function': func_key,
                        'start_time': start_time,
                        'end_time': entry.timestamp,
                        'duration_seconds': duration,
                        'message': entry.message
                    })
                    del active_functions[func_key]
        
        df = pd.DataFrame(function_durations)
        if not df.empty:
            # 按函数分组，计算总用时和平均用时
            summary = df.groupby('function').agg({
                'duration_seconds': ['sum', 'mean', 'count', 'max', 'min']
            }).round(2)
            summary.columns = ['total_seconds', 'avg_seconds', 'call_count', 'max_seconds', 'min_seconds']
            summary = summary.sort_values('total_seconds', ascending=False)
            return summary
        return pd.DataFrame()
    
    def analyze_module_timing(self) -> pd.DataFrame:
        """分析每个模块的总用时"""
        module_times = defaultdict(list)
        
        for entry in self.entries:
            if entry.module:
                module_times[entry.module].append(entry.timestamp)
        
        results = []
        for module, timestamps in module_times.items():
            if len(timestamps) > 1:
                duration = (max(timestamps) - min(timestamps)).total_seconds()
                results.append({
                    'module': module,
                    'duration_seconds': duration,
                    'log_count': len(timestamps)
                })
        
        df = pd.DataFrame(results)
        df = df.sort_values('duration_seconds', ascending=False)
        return df

# 执行时间分析
time_analyzer = TimeAnalyzer(entries)

# 分析总用时
total_duration = parser.get_total_duration()
print(f"总用时: {total_duration:.2f} 秒 ({total_duration/3600:.2f} 小时)")
if len(entries) > 0:
    print(f"开始时间: {entries[0].timestamp}")
    print(f"结束时间: {entries[-1].timestamp}")
else:
    print("警告: 日志文件为空，无法显示时间范围")
print()

总用时: 2956.19 秒 (0.82 小时)
开始时间: 2026-02-02 18:29:42.367732
结束时间: 2026-02-02 19:18:58.556086



In [71]:
# # 分析函数执行时间（通过开始/完成模式）
# function_timing_df = time_analyzer.analyze_function_duration_by_pattern()
# print("=== 函数执行时间分析（通过开始/完成模式）===")
# print(f"共找到 {len(function_timing_df)} 个函数的时间记录")
# print()
# print("Top 20 函数（按总用时排序）:")
# # 使用 to_string() 确保显示完整的函数名
# print(function_timing_df.head(20).to_string())

# # 分析模块总用时
# module_timing_df = time_analyzer.analyze_module_timing()
# print("=== 模块总用时分析 ===")
# print(f"共找到 {len(module_timing_df)} 个模块")
# print()
# print("Top 15 模块（按总用时排序）:")
# # 使用 to_string() 确保显示完整的模块名
# print(module_timing_df.head(15).to_string())


## 3. is_answerable=False 问题分析模块

分析所有最后输出为 `is_answerable=False` 的问题的路径和原因


In [72]:
class UnanswerableQuestionAnalyzer:
    """不可回答问题分析器"""
    def __init__(self, entries: List[LogEntry]):
        self.entries = entries
    
    def find_unanswerable_questions(self) -> List[Dict]:
        """查找所有 is_answerable=False 的问题"""
        unanswerable_questions = []
        
        for entry in self.entries:
            if entry.message and 'is_answerable=False' in entry.message:
                # 尝试提取问题信息
                question_info = self._extract_question_info(entry)
                if question_info:
                    unanswerable_questions.append(question_info)
        
        return unanswerable_questions
    
    def _extract_question_info(self, entry: LogEntry) -> Optional[Dict]:
        """从日志条目中提取问题信息"""
        try:
            # 查找问题ID和问题文本
            question_id_match = re.search(r"question_id['\"]?\s*:\s*['\"]?([^'\",\s}]+)", entry.message)
            question_text_match = re.search(r"question_text['\"]?\s*:\s*['\"]?([^'\"]+)", entry.message)
            answer_text_match = re.search(r"answer_text['\"]?\s*:\s*['\"]?([^'\"]+)", entry.message)
            
            question_id = question_id_match.group(1) if question_id_match else None
            question_text = question_text_match.group(1) if question_text_match else None
            answer_text = answer_text_match.group(1) if answer_text_match else None
            
            if question_id:
                return {
                    'question_id': question_id,
                    'question_text': question_text,
                    'answer_text': answer_text,
                    'timestamp': entry.timestamp,
                    'module': entry.module,
                    'function': entry.function,
                    'full_message': entry.message
                }
        except Exception as e:
            pass
        
        return None
    
    def analyze_reasons(self, questions: List[Dict]) -> pd.DataFrame:
        """分析不可回答的原因"""
        reasons = []
        
        for q in questions:
            # 确保 answer_text 是字符串类型，处理 None 的情况
            answer_text = q.get('answer_text') or ''
            if answer_text is None:
                answer_text = ''
            else:
                answer_text = str(answer_text)
            
            # 提取原因关键词
            reason_keywords = []
            
            if answer_text and '信息不足' in answer_text:
                reason_keywords.append('信息不足')
            if answer_text and '无法确定' in answer_text:
                reason_keywords.append('无法确定')
            if answer_text and '无法回答' in answer_text:
                reason_keywords.append('无法回答')
            if answer_text and '缺乏' in answer_text:
                reason_keywords.append('缺乏数据')
            
            reasons.append({
                'question_id': q.get('question_id', ''),
                'question_text': q.get('question_text', ''),
                'reason': '; '.join(reason_keywords) if reason_keywords else '其他',
                'answer_text': answer_text[:200] if answer_text else ''  # 截断长文本
            })
        
        # 确保即使列表为空，也返回带有正确列名的 DataFrame
        if not reasons:
            return pd.DataFrame(columns=['question_id', 'question_text', 'reason', 'answer_text'])
        
        return pd.DataFrame(reasons)
    
    def trace_question_path(self, question_id: str) -> List[LogEntry]:
        """追踪问题的处理路径"""
        path = []
        for entry in self.entries:
            if entry.message and question_id in entry.message:
                path.append(entry)
        return path

# 分析不可回答问题
unanswerable_analyzer = UnanswerableQuestionAnalyzer(entries)
unanswerable_questions = unanswerable_analyzer.find_unanswerable_questions()

print(f"=== 不可回答问题分析 ===")
print(f"共找到 {len(unanswerable_questions)} 个 is_answerable=False 的问题")
print()

=== 不可回答问题分析 ===
共找到 0 个 is_answerable=False 的问题



In [73]:
# 分析原因
reasons_df = unanswerable_analyzer.analyze_reasons(unanswerable_questions)
print("=== 不可回答原因统计 ===")
if len(reasons_df) > 0:
    reason_counts = reasons_df['reason'].value_counts()
    print(reason_counts)
    print()
    
    # 显示详细列表
    print("=== 所有不可回答问题的详细信息 ===")
    display_df = reasons_df[['question_id', 'question_text', 'reason']].copy()
    display_df.index = range(1, len(display_df) + 1)
    print(display_df.to_string())
else:
    print("没有找到不可回答的问题")
    print()


=== 不可回答原因统计 ===
没有找到不可回答的问题



## 4. ERROR 日志分析模块

总结和分析所有 `logger.error` 的产生和原因


In [74]:
class ErrorAnalyzer:
    """错误日志分析器"""
    def __init__(self, entries: List[LogEntry]):
        self.entries = entries
    
    def find_errors(self) -> List[LogEntry]:
        """查找所有 ERROR 级别的日志"""
        return [e for e in self.entries if e.level == 'ERROR']
    
    def categorize_errors(self, error_entries: List[LogEntry]) -> pd.DataFrame:
        """对错误进行分类"""
        error_categories = []
        
        for entry in error_entries:
            category = self._categorize_error(entry.message or '')
            error_categories.append({
                'timestamp': entry.timestamp,
                'module': entry.module,
                'function': entry.function,
                'category': category,
                'message': entry.message[:200] if entry.message else '',  # 截断长文本
                'full_message': entry.message
            })
        
        return pd.DataFrame(error_categories)
    
    def _categorize_error(self, message: str) -> str:
        """根据错误消息分类"""
        message_lower = message.lower()
        
        if 'context length' in message_lower or 'maximum context' in message_lower:
            return '上下文长度超限'
        elif 'timeout' in message_lower:
            return '超时错误'
        elif 'connection' in message_lower or 'connect' in message_lower:
            return '连接错误'
        elif 'invalid' in message_lower or '无效' in message_lower:
            return '无效参数/数据'
        elif 'failed' in message_lower or '失败' in message_lower:
            return '执行失败'
        elif 'exception' in message_lower or '异常' in message_lower:
            return '异常错误'
        elif 'unsupported' in message_lower or '不支持' in message_lower:
            return '不支持的操作'
        elif 'tracing' in message_lower or 'trace' in message_lower:
            return '追踪错误'
        else:
            return '其他错误'
    
    def analyze_error_patterns(self, error_df: pd.DataFrame) -> Dict:
        """分析错误模式"""
        patterns = {
            'total_errors': len(error_df),
            'by_category': error_df['category'].value_counts().to_dict(),
            'by_module': error_df['module'].value_counts().to_dict(),
            'by_function': error_df['function'].value_counts().to_dict(),
            'time_distribution': self._analyze_error_time_distribution(error_df)
        }
        return patterns
    
    def _analyze_error_time_distribution(self, error_df: pd.DataFrame) -> Dict:
        """分析错误的时间分布"""
        if error_df.empty:
            return {}
        
        error_df['hour'] = error_df['timestamp'].dt.hour
        hour_counts = error_df['hour'].value_counts().sort_index()
        return hour_counts.to_dict()

# 分析错误日志
error_analyzer = ErrorAnalyzer(entries)
error_entries = error_analyzer.find_errors()

print(f"=== ERROR 日志分析 ===")
print(f"共找到 {len(error_entries)} 条 ERROR 日志")
print()


=== ERROR 日志分析 ===
共找到 36 条 ERROR 日志



In [75]:
# 错误分类和分析
if error_entries:
    error_df = error_analyzer.categorize_errors(error_entries)
    error_patterns = error_analyzer.analyze_error_patterns(error_df)
    
    print("=== 错误分类统计 ===")
    print(f"总错误数: {error_patterns['total_errors']}")
    print()
    
    print("按类别统计:")
    for category, count in error_patterns['by_category'].items():
        print(f"  {category}: {count}")
    print()
    
    print("按模块统计 (Top 10):")
    module_counts = error_patterns['by_module']
    for module, count in list(module_counts.items())[:10]:
        print(f"  {module}: {count}")
    print()
    
    print("按函数统计 (Top 10):")
    function_counts = error_patterns['by_function']
    for func, count in list(function_counts.items())[:10]:
        print(f"  {func}: {count}")
    print()
    
    # 显示详细错误列表
    print("=== 所有错误详细信息 ===")
    display_error_df = error_df[['timestamp', 'module', 'function', 'category', 'message']].copy()
    display_error_df.index = range(1, len(display_error_df) + 1)
    print(display_error_df.to_string())
else:
    print("未找到 ERROR 级别的日志")


=== 错误分类统计 ===
总错误数: 36

按类别统计:
  其他错误: 35
  执行失败: 1

按模块统计 (Top 10):
  src.agents.content_generate_agent.retrieval_workflow.context_reduction_utils: 35
  src.applications.content_generate.routers.content_generate_router: 1

按函数统计 (Top 10):
  filter_selected_content_for_single_question: 35
  _send_callback: 1

=== 所有错误详细信息 ===
                    timestamp                                                                        module                                     function category                                                                                                                                                                                                   message
1  2026-02-02 19:04:13.320822  src.agents.content_generate_agent.retrieval_workflow.context_reduction_utils  filter_selected_content_for_single_question     其他错误                                                                                                                                                  

In [76]:
# 显示典型错误的详细信息
if error_entries:
    error_df = error_analyzer.categorize_errors(error_entries)
    
    print("=== 典型错误详细信息 ===")
    # 按类别分组，每个类别显示一个示例
    for category in error_df['category'].unique():
        sample = error_df[error_df['category'] == category].iloc[0]
        print(f"\n类别: {category}")
        print(f"时间: {sample['timestamp']}")
        print(f"模块: {sample['module']}")
        print(f"函数: {sample['function']}")
        print(f"消息: {sample['full_message'][:300]}...")
        print("-" * 80)


=== 典型错误详细信息 ===

类别: 其他错误
时间: 2026-02-02 19:04:13.320822
模块: src.agents.content_generate_agent.retrieval_workflow.context_reduction_utils
函数: filter_selected_content_for_single_question
消息: [筛选] piece (doc_id=chapter_74_part12) 无法获取文本内容，已跳过...
--------------------------------------------------------------------------------

类别: 执行失败
时间: 2026-02-02 19:07:45.869931
模块: src.applications.content_generate.routers.content_generate_router
函数: _send_callback
消息: 回调请求失败: callback_payload={'data': '{"questionId": "q_66", "questionText": "需要总结该公司在行业风险、盈利能力、应收账款坏账风险、资本支出压力及负面信息等方面存在的主要问题，包括但不限于行业景气度、毛利率变化、减值损失增长、应收账款结构与回款风险、在建项目资本支出压力、被执行记录、资金冻结情况及行政处罚等，需基于财务数据、经营状况和外部环境综合分析。", "questionType": "text", "answer": {"isAnswerable": true, "answerText": "该公司在行业风险、盈利能...
--------------------------------------------------------------------------------


## 5. 问题处理路径分析模块

分析每个问题在处理过程中的路径，包括：
1. 在经过 chunk_index_retrieval 后第一次进入 judge_if_continue 节点时是否被认为"内容充足"
2. 在经过 summary_index_retrieval 后第二次进入 judge_if_continue 节点后是否被认为内容充足
3. 最后一次是否被认为内容充足


In [77]:
class QuestionPathAnalyzer:
    """问题处理路径分析器
    
    使用新的日志格式 [问题路径分析] 直接提取判断结果，简化分析逻辑
    """
    def __init__(self, entries: List[LogEntry]):
        self.entries = entries
    
    def analyze_question_paths(self) -> pd.DataFrame:
        """分析每个问题的处理路径
        
        使用新的日志格式：
        [问题路径分析] question_id={qid}, 阶段={阶段}后, 判断结果={结果}, 原因={原因}
        
        直接通过正则表达式提取，无需复杂的阶段判断逻辑
        """
        # 正则表达式模式：提取 [问题路径分析] 日志中的信息
        pattern = r'\[问题路径分析\] question_id=([^,]+),\s*阶段=([^后]+)后,\s*判断结果=([^,]+)(?:,\s*原因=(.+))?'
        
        # 存储每个问题的判断结果
        question_states = {}
        
        # 遍历所有日志条目，提取 [问题路径分析] 信息
        for entry in self.entries:
            if entry.message and '[问题路径分析]' in entry.message:
                match = re.search(pattern, entry.message)
                if match:
                    qid = match.group(1).strip()
                    phase = match.group(2).strip()
                    result = match.group(3).strip()
                    reason = match.group(4).strip() if match.group(4) else ''
                    
                    # 初始化问题状态（如果还没有）
                    if qid not in question_states:
                        question_states[qid] = {
                            'question_id': qid,
                            'first_judge_after_chunk': None,
                            'second_judge_after_summary': None,
                            'final_judge': None,
                            'first_judge_timestamp': None,
                            'second_judge_timestamp': None,
                            'final_judge_timestamp': None,
                        }
                    
                    # 根据阶段和结果更新状态
                    is_sufficient = None
                    if result == '充足' or result == '充足(已完成)':
                        is_sufficient = True
                    elif result == '不足' or result == '不足(无内容)':
                        is_sufficient = False
                    # result == '错误' 时不更新状态
                    
                    if is_sufficient is not None:
                        if phase == 'chunk':
                            if question_states[qid]['first_judge_after_chunk'] is None:
                                question_states[qid]['first_judge_after_chunk'] = is_sufficient
                                question_states[qid]['first_judge_timestamp'] = entry.timestamp
                        elif phase == 'summary':
                            if question_states[qid]['second_judge_after_summary'] is None:
                                question_states[qid]['second_judge_after_summary'] = is_sufficient
                                question_states[qid]['second_judge_timestamp'] = entry.timestamp
                        elif phase == 'document_reading':
                            if question_states[qid]['final_judge'] is None:
                                question_states[qid]['final_judge'] = is_sufficient
                                question_states[qid]['final_judge_timestamp'] = entry.timestamp
        
        # 收集所有问题ID（包括从其他日志中提取的）
        all_question_ids = set(question_states.keys())
        
        # 从其他日志中提取问题ID（作为补充）
        for entry in self.entries:
            if entry.message:
                # 从 "开始评估问题: question_id=#13" 中提取
                qid_matches = re.findall(r"question_id=([^\s,]+)", entry.message)
                for qid in qid_matches:
                    all_question_ids.add(qid.strip())
                
                # 从 finished_question_ids 中提取
                if 'finished_question_ids' in entry.message or '检测到finished_question_ids' in entry.message:
                    ids_match = re.search(r"\[(.*?)\]", entry.message)
                    if ids_match:
                        ids_str = ids_match.group(1)
                        finished_ids = re.findall(r"#\d+", ids_str)
                        for fid in finished_ids:
                            all_question_ids.add(fid)
        
        # 为没有 [问题路径分析] 日志的问题初始化状态
        for qid in all_question_ids:
            if qid not in question_states:
                question_states[qid] = {
                    'question_id': qid,
                    'first_judge_after_chunk': None,
                    'second_judge_after_summary': None,
                    'final_judge': None,
                    'first_judge_timestamp': None,
                    'second_judge_timestamp': None,
                    'final_judge_timestamp': None,
                }
        
        print(f"找到 {len(all_question_ids)} 个问题ID")
        
        # 转换为 DataFrame
        results = []
        for qid, state in question_states.items():
            results.append({
                'question_id': qid,
                '第一次判断(chunk后)': '充足' if state['first_judge_after_chunk'] else ('不足' if state['first_judge_after_chunk'] is False else '未判断'),
                '第二次判断(summary后)': '充足' if state['second_judge_after_summary'] else ('不足' if state['second_judge_after_summary'] is False else '未判断'),
                '最后一次判断': '充足' if state['final_judge'] else ('不足' if state['final_judge'] is False else '未判断'),
                '第一次判断时间': state['first_judge_timestamp'],
                '第二次判断时间': state['second_judge_timestamp'],
                '最后一次判断时间': state['final_judge_timestamp'],
            })
        
        df = pd.DataFrame(results)
        return df
    
    def analyze_path_statistics(self, path_df: pd.DataFrame) -> Dict:
        """分析路径统计信息"""
        stats = {
            'total_questions': len(path_df),
            'first_judge_sufficient': len(path_df[path_df['第一次判断(chunk后)'] == '充足']),
            'first_judge_insufficient': len(path_df[path_df['第一次判断(chunk后)'] == '不足']),
            'second_judge_sufficient': len(path_df[path_df['第二次判断(summary后)'] == '充足']),
            'second_judge_insufficient': len(path_df[path_df['第二次判断(summary后)'] == '不足']),
            'final_judge_sufficient': len(path_df[path_df['最后一次判断'] == '充足']),
            'final_judge_insufficient': len(path_df[path_df['最后一次判断'] == '不足']),
        }
        
        # 计算路径分布
        path_df['path'] = path_df.apply(lambda row: self._get_path_string(row), axis=1)
        path_distribution = path_df['path'].value_counts().to_dict()
        stats['path_distribution'] = path_distribution
        
        return stats
    
    def _get_path_string(self, row) -> str:
        """获取路径字符串"""
        first = row['第一次判断(chunk后)']
        second = row['第二次判断(summary后)']
        final = row['最后一次判断']
        
        if first == '充足':
            return 'chunk后充足→直接完成'
        elif first == '不足' and second == '充足':
            return 'chunk后不足→summary后充足→完成'
        elif first == '不足' and second == '不足' and final == '充足':
            return 'chunk后不足→summary后不足→document_reading后充足→完成'
        elif first == '不足' and second == '不足' and final == '不足':
            return 'chunk后不足→summary后不足→document_reading后不足'
        elif first == '不足' and second == '未判断' and final == '充足':
            return 'chunk后不足→document_reading后充足→完成'
        elif first == '不足' and second == '未判断' and final == '不足':
            return 'chunk后不足→document_reading后不足'
        else:
            return '其他路径'

# 执行问题路径分析
path_analyzer = QuestionPathAnalyzer(entries)
path_df = path_analyzer.analyze_question_paths()

print(f"=== 问题处理路径分析 ===")
print(f"共分析 {len(path_df)} 个问题")
print()


找到 49 个问题ID
=== 问题处理路径分析 ===
共分析 49 个问题



In [78]:
# 分析路径统计
path_stats = path_analyzer.analyze_path_statistics(path_df)

print("=== 路径统计信息 ===")
print(f"总问题数: {path_stats['total_questions']}")
print()
print("第一次判断(chunk后):")
print(f"  内容充足: {path_stats['first_judge_sufficient']} 个")
print(f"  内容不足: {path_stats['first_judge_insufficient']} 个")
print()
print("第二次判断(summary后):")
print(f"  内容充足: {path_stats['second_judge_sufficient']} 个")
print(f"  内容不足: {path_stats['second_judge_insufficient']} 个")
print()
print("最后一次判断:")
print(f"  内容充足: {path_stats['final_judge_sufficient']} 个")
print(f"  内容不足: {path_stats['final_judge_insufficient']} 个")
print()
print("=== 路径分布 ===")
for path, count in sorted(path_stats['path_distribution'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {path}: {count} 个问题")
print()


=== 路径统计信息 ===
总问题数: 49

第一次判断(chunk后):
  内容充足: 23 个
  内容不足: 26 个

第二次判断(summary后):
  内容充足: 20 个
  内容不足: 6 个

最后一次判断:
  内容充足: 0 个
  内容不足: 0 个

=== 路径分布 ===
  chunk后充足→直接完成: 23 个问题
  chunk后不足→summary后充足→完成: 20 个问题
  其他路径: 6 个问题



In [79]:
# 调试信息：显示一些示例问题的详细路径，帮助检查代码逻辑
print("=== 调试信息：示例问题路径 ===")
if len(path_df) > 0:
    # 显示前5个问题的详细信息
    sample_df = path_df.head(5).copy()
    for idx, row in sample_df.iterrows():
        print(f"\n问题ID: {row['question_id']}")
        print(f"  第一次判断(chunk后): {row['第一次判断(chunk后)']}")
        print(f"  第二次判断(summary后): {row['第二次判断(summary后)']}")
        print(f"  最后一次判断: {row['最后一次判断']}")
        print(f"  第一次判断时间: {row['第一次判断时间']}")
        print(f"  第二次判断时间: {row['第二次判断时间']}")
        print(f"  最后一次判断时间: {row['最后一次判断时间']}")
else:
    print("未找到任何问题路径数据")
print()


=== 调试信息：示例问题路径 ===

问题ID: q_1
  第一次判断(chunk后): 不足
  第二次判断(summary后): 充足
  最后一次判断: 未判断
  第一次判断时间: 2026-02-02 18:45:53.401544
  第二次判断时间: 2026-02-02 18:49:04.515918
  最后一次判断时间: None

问题ID: q_2
  第一次判断(chunk后): 不足
  第二次判断(summary后): 充足
  最后一次判断: 未判断
  第一次判断时间: 2026-02-02 18:45:53.402788
  第二次判断时间: 2026-02-02 18:49:04.516835
  最后一次判断时间: None

问题ID: q_3
  第一次判断(chunk后): 不足
  第二次判断(summary后): 不足
  最后一次判断: 未判断
  第一次判断时间: 2026-02-02 18:45:53.403831
  第二次判断时间: 2026-02-02 18:49:04.518544
  最后一次判断时间: None

问题ID: q_4
  第一次判断(chunk后): 不足
  第二次判断(summary后): 不足
  最后一次判断: 未判断
  第一次判断时间: 2026-02-02 18:45:53.404371
  第二次判断时间: 2026-02-02 18:49:04.519748
  最后一次判断时间: None

问题ID: q_5
  第一次判断(chunk后): 不足
  第二次判断(summary后): 不足
  最后一次判断: 未判断
  第一次判断时间: 2026-02-02 18:45:53.405344
  第二次判断时间: 2026-02-02 18:49:04.520642
  最后一次判断时间: None



In [80]:
# 显示详细的问题路径信息
print("=== 所有问题的详细路径信息 ===")
# 只显示关键列
display_path_df = path_df[['question_id', '第一次判断(chunk后)', '第二次判断(summary后)', '最后一次判断']].copy()
display_path_df.index = range(1, len(display_path_df) + 1)
print(display_path_df.to_string())

=== 所有问题的详细路径信息 ===
   question_id 第一次判断(chunk后) 第二次判断(summary后) 最后一次判断
1          q_1            不足              充足    未判断
2          q_2            不足              充足    未判断
3          q_3            不足              不足    未判断
4          q_4            不足              不足    未判断
5          q_5            不足              不足    未判断
6          q_6            不足              不足    未判断
7          q_7            充足             未判断    未判断
8          q_8            不足              充足    未判断
9          q_9            不足              充足    未判断
10        q_10            不足              充足    未判断
11        q_11            充足             未判断    未判断
12        q_12            充足             未判断    未判断
13        q_13            充足             未判断    未判断
14        q_14            充足             未判断    未判断
15        q_15            充足             未判断    未判断
16        q_16            充足             未判断    未判断
17        q_17            充足             未判断    未判断
18        q_18            充足             未判断

## 6. 问题答案提取模块

提取每个问题的答案（从FinalAnswer中），构造 question_id, question_text, question_answer 三元组


In [81]:
class AnswerExtractor:
    """问题答案提取器
    
    从日志中提取每个问题的答案（适配新格式）
    """
    def __init__(self, entries: List[LogEntry]):
        self.entries = entries
    
    def extract_answers(self) -> pd.DataFrame:
        """提取所有问题的答案
        
        从日志中提取格式（新格式）：
        处理问题: q_6 完成，输出结果: {'questionId': 'q_6', 'questionText': '...', 'questionType': 'text', 'answer': {'isAnswerable': False, 'answerText': '...', 'citations': [], 'thinkingSteps': {...}}}
        """
        answers = []
        
        # 统计信息
        total_checked = 0
        found_entries = 0
        
        for entry in self.entries:
            if not entry.message:
                continue
            
            total_checked += 1
            
            # 检查是否包含关键信息
            if '处理问题:' not in entry.message:
                continue
            if '完成，输出结果:' not in entry.message:
                continue
            
            found_entries += 1
            
            # 尝试从消息中提取字典字符串
            # 查找 "输出结果: " 后面的字典
            result_match = re.search(r"输出结果:\s*(\{.*\})", entry.message, re.DOTALL)
            if not result_match:
                continue
            
            result_str = result_match.group(1)
            
            # 尝试解析为字典
            result_dict = None
            try:
                # 先尝试直接使用 ast.literal_eval（如果字典使用双引号）
                import ast
                result_dict = ast.literal_eval(result_str)
            except (ValueError, SyntaxError):
                # 如果失败，尝试将单引号转换为双引号（但需要小心处理字符串内的引号）
                try:
                    # 使用 json.loads，但需要先将单引号转换为双引号
                    # 注意：这个方法可能不完美，因为字符串内可能包含引号
                    import json
                    # 简单的转换：将外层的单引号改为双引号
                    json_str = result_str.replace("'", '"')
                    result_dict = json.loads(json_str)
                except (ValueError, json.JSONDecodeError):
                    # 如果都失败，使用正则表达式提取
                    result_dict = None
            
            # 提取 questionId 和 questionText
            qid = None
            qtext = None
            answer_text = ""
            
            if result_dict and isinstance(result_dict, dict):
                # 从字典中提取
                qid = result_dict.get('questionId') or result_dict.get('question_id')
                qtext = result_dict.get('questionText') or result_dict.get('question_text')
                answer_dict = result_dict.get('answer', {})
                if isinstance(answer_dict, dict):
                    answer_text = answer_dict.get('answerText') or answer_dict.get('answer_text', '')
            else:
                # 如果解析失败，使用正则表达式提取
                # 提取 questionId (支持 'questionId' 和 'question_id')
                qid_match = re.search(r"['\"]questionId['\"]\s*:\s*['\"]([^'\"]+)['\"]", result_str)
                if not qid_match:
                    qid_match = re.search(r"['\"]question_id['\"]\s*:\s*['\"]?([^'\",\s}]+)", result_str)
                if qid_match:
                    qid = qid_match.group(1).strip()
                
                # 提取 questionText (支持 'questionText' 和 'question_text')
                qtext_match = re.search(r"['\"]questionText['\"]\s*:\s*['\"]([^'\"]+)['\"]", result_str)
                if not qtext_match:
                    qtext_match = re.search(r"['\"]question_text['\"]\s*:\s*['\"]?([^'\"]+)", result_str)
                if qtext_match:
                    qtext = qtext_match.group(1).strip()
                
                # 提取 answerText (从 answer 字典中)
                # 查找 'answer': {'answerText': '...', ...}
                # 注意：answerText 可能包含多行和特殊字符，需要匹配到 ', 'citations' 或 ', 'thinkingSteps' 之前
                answer_match = re.search(r"['\"]answerText['\"]\s*:\s*['\"](.*?)(?:['\"],\s*['\"]citations)", result_str, re.DOTALL)
                if not answer_match:
                    # 如果找不到 ', 'citations'，尝试匹配到 ', 'thinkingSteps' 之前
                    answer_match = re.search(r"['\"]answerText['\"]\s*:\s*['\"](.*?)(?:['\"],\s*['\"]thinkingSteps)", result_str, re.DOTALL)
                if not answer_match:
                    # 如果还是找不到，尝试匹配到 '}' 之前（answer 字典结束）
                    answer_match = re.search(r"['\"]answerText['\"]\s*:\s*['\"](.*?)(?:['\"]\s*\})", result_str, re.DOTALL)
                if answer_match:
                    answer_text = answer_match.group(1).strip()
            
            if not qid or not qtext:
                continue
            
            # 处理转义字符
            if answer_text:
                answer_text = answer_text.replace('\\n', '\n').replace('\\r', '\r').replace('\\t', '\t')
                answer_text = answer_text.replace("\\'", "'").replace('\\"', '"')
            
            # 清理答案文本
            if answer_text:
                # 移除多余的空白字符（但保留换行）
                lines = answer_text.split('\n')
                answer_text = '\n'.join(line.strip() for line in lines if line.strip())
            
            # 生成 answer_raw：去掉所有[]中的引用标记
            answer_raw = answer_text
            if answer_raw:
                # 使用正则表达式去掉所有[...]格式的引用标记
                answer_raw = re.sub(r'\[[^\]]*\]', '', answer_raw)
                # 清理多余的空白字符
                answer_raw = ' '.join(answer_raw.split())
            
            # 只有成功提取到所有信息才添加
            if qid and qtext:
                answers.append({
                    'question_id': qid,
                    'question_text': qtext,
                    'question_answer': answer_text,
                    'answer_raw': answer_raw,
                    'timestamp': entry.timestamp
                })
        
        print(f"调试信息: 检查了 {total_checked} 条日志，找到 {found_entries} 条包含输出结果的日志，成功提取 {len(answers)} 个答案")
        
        df = pd.DataFrame(answers)
        # 按 question_id 去重，保留最新的答案
        if not df.empty:
            # 统计去重前的情况
            question_counts = df['question_id'].value_counts()
            duplicate_questions = question_counts[question_counts > 1]
            
            if len(duplicate_questions) > 0:
                print(f"\n调试信息: 发现 {len(duplicate_questions)} 个问题有重复答案（共 {duplicate_questions.sum()} 个重复答案）")
                print("重复问题详情:")
                for qid, count in duplicate_questions.items():
                    qtext = df[df['question_id'] == qid]['question_text'].iloc[0]
                    timestamps = df[df['question_id'] == qid]['timestamp'].tolist()
                    print(f"  {qid} ({qtext}): {count} 次")
                    print(f"    时间戳: {[str(ts) for ts in timestamps]}")
            
            # 去重，保留最新的答案
            df_before_dedup = len(df)
            df = df.sort_values('timestamp').drop_duplicates(subset=['question_id'], keep='last')
            df_after_dedup = len(df)
            df = df.sort_values('question_id').reset_index(drop=True)
            
            print(f"\n调试信息: 去重前 {df_before_dedup} 个答案，去重后 {df_after_dedup} 个唯一问题")
        else:
            # 如果没提取到，尝试打印一个示例看看格式
            if found_entries > 0:
                print("\n调试信息: 找到包含输出结果的日志，但提取失败。示例日志（前1000字符）：")
                for entry in self.entries:
                    if entry.message and '处理问题:' in entry.message and '完成，输出结果:' in entry.message:
                        print(entry.message[:1000])
                        break
        
        return df

# 执行答案提取
answer_extractor = AnswerExtractor(entries)
answers_df = answer_extractor.extract_answers()

print(f"=== 问题答案提取 ===")
print(f"共提取 {len(answers_df)} 个问题的答案")
print()

# 显示前5个示例
if len(answers_df) > 0:
    print("=== 示例答案（前5个）===")
    for idx, row in answers_df.head(5).iterrows():
        print(f"\n问题ID: {row['question_id']}")
        print(f"问题文本: {row['question_text']}")
        answer_preview = row['question_answer'][:200] + "..." if len(row['question_answer']) > 200 else row['question_answer']
        print(f"答案(含引用): {answer_preview}")
        answer_raw_preview = row['answer_raw'][:200] + "..." if len(row['answer_raw']) > 200 else row['answer_raw']
        print(f"答案(去引用): {answer_raw_preview}")
else:
    print("未找到任何答案")
print()


调试信息: 检查了 6498 条日志，找到 75 条包含输出结果的日志，成功提取 75 个答案

调试信息: 去重前 75 个答案，去重后 75 个唯一问题
=== 问题答案提取 ===
共提取 75 个问题的答案

=== 示例答案（前5个）===

问题ID: q_1
问题文本: 需要填写提交该授信初审报告的呈报单位名称，通常为分行或分支机构的全称，应根据实际负责该业务的机构信息进行填写。
答案(含引用): 提交该授信初审报告的呈报单位名称应为北京银行脱敏B分行，因为该报告由北京银行脱敏B分行开展尽职调查并作为信贷审批的基础资料，且报告中明确提及调查机构为‘北京银行脱敏B分行’[1]，调查结论亦由该分行出具并纳入其集团客户授信管理[2]。
答案(去引用): 提交该授信初审报告的呈报单位名称应为北京银行脱敏B分行，因为该报告由北京银行脱敏B分行开展尽职调查并作为信贷审批的基础资料，且报告中明确提及调查机构为‘北京银行脱敏B分行’，调查结论亦由该分行出具并纳入其集团客户授信管理。

问题ID: q_10
问题文本: 判断该公司是否属于政府融资平台（包括监管类或监测类），需根据企业股权结构、实际控制人、融资用途及政府相关文件等信息进行核实。
答案(含引用): 该公司不属于政府融资平台。根据股权结构，中国铁建股份有限公司的控股股东为中国铁道建筑集团有限公司，而中国铁道建筑集团有限公司的实际控制人为国务院国有资产监督管理委员会（国资委）[1][2]，表明该公司为中央企业，而非地方政府融资平台。其融资用途主要用于补充流动资金、基础设施建设等经营性用途，且募集资金使用符合规定，未用于地方政府隐性债务[3][4]。此外，公司发行的债券（如铁建YK系列）均计入其他...
答案(去引用): 该公司不属于政府融资平台。根据股权结构，中国铁建股份有限公司的控股股东为中国铁道建筑集团有限公司，而中国铁道建筑集团有限公司的实际控制人为国务院国有资产监督管理委员会（国资委），表明该公司为中央企业，而非地方政府融资平台。其融资用途主要用于补充流动资金、基础设施建设等经营性用途，且募集资金使用符合规定，未用于地方政府隐性债务。此外，公司发行的债券（如铁建YK系列）均计入其他权益工具，属于企业自主融...

问题ID: q_11
问题文本: 判断该公司是否为银行的关联方，需根据企业股权结构、管理层

In [82]:
# 保存为 JSON 文件
if len(answers_df) > 0:
    # 在保存和输出前，按照 question_id 进行排序
    # 提取 question_id 中的数字部分用于排序
    def extract_number(qid):
        """从 question_id 中提取数字部分，用于排序"""
        if isinstance(qid, str) and qid.startswith('#'):
            try:
                return int(qid[1:])
            except ValueError:
                return 0
        return 0
    
    # 添加临时排序列
    answers_df['_sort_key'] = answers_df['question_id'].apply(extract_number)
    # 按数字排序
    answers_df = answers_df.sort_values('_sort_key').reset_index(drop=True)
    # 删除临时列
    answers_df = answers_df.drop(columns=['_sort_key'])
    
    # 转换为字典列表格式（包含 answer_raw 字段）
    answers_list = answers_df[['question_id', 'question_text', 'question_answer', 'answer_raw']].to_dict('records')
    
    # 保存路径
    import json
    from pathlib import Path
    
    output_dir = Path("../storage/testlog")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # 使用 TEST_VER 作为输出文件名
    output_file = output_dir / f"{TEST_VER}test_answers.json"
    
    # 保存为 JSON
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(answers_list, f, ensure_ascii=False, indent=2)
    
    print(f"=== 答案已保存 ===")
    print(f"保存路径: {output_file}")
    print(f"共保存 {len(answers_list)} 个问题的答案")
    print()
    
    # 显示 DataFrame
    print("=== 所有问题的答案（DataFrame）===")
    display_df = answers_df[['question_id', 'question_text', 'question_answer', 'answer_raw']].copy()
    display_df.index = range(1, len(display_df) + 1)
    print(display_df.to_string())
else:
    print("没有答案可保存")


=== 答案已保存 ===
保存路径: ..\storage\testlog\0202test_answers.json
共保存 75 个问题的答案

=== 所有问题的答案（DataFrame）===
   question_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 question_text                                                                                                                                                                                                                                                                      

## 7.上下文提取模块

从Langfuse中提取纪录并导出每个问题的上下文`content_text`

In [83]:
# import json
# from pathlib import Path
# from datetime import datetime
# import pandas as pd
# from langfuse.api.client import FernLangfuse

# HOST = "http://10.192.72.11:3111"
# PK = "pk-lf-5a2d9428-8511-4bf2-839e-5c6f1d8c32d5"
# SK = "sk-lf-0d86e243-0cf3-4958-9fcd-7b26d75755ed"

# NAME_KEYWORD = "reporter"  # 生成阶段 observation 名称关键词

# # 日志时间窗口（entries 已在前面解析）
# try:
#     start_ts = entries[0].timestamp if entries else None
#     end_ts = entries[-1].timestamp if entries else None
# except Exception:
#     start_ts = end_ts = None

# def to_local_naive(dt: datetime):
#     """将 datetime 转换为本地 naive 时间（用于比较）"""
#     if not dt:
#         return None
#     if dt.tzinfo:
#         dt = dt.astimezone().replace(tzinfo=None)
#     return dt

# def parse_ts(val):
#     """解析时间戳，支持多种格式"""
#     if not val:
#         return None
#     if isinstance(val, (int, float)):
#         return datetime.fromtimestamp(val/1000 if val > 1e12 else val)
#     if isinstance(val, str):
#         s = val.replace('Z', '+00:00')
#         for fmt in (datetime.fromisoformat, lambda x: datetime.strptime(x, '%Y-%m-%dT%H:%M:%S.%fZ')):
#             try:
#                 return fmt(s)
#             except Exception:
#                 continue
#     return None

# def extract_context(obs):
#     """从 observation 中提取上下文和问题"""
#     # 处理 SDK 返回的对象（可能是 Pydantic 模型或字典）
#     if hasattr(obs, 'input'):
#         inp = obs.input
#     elif isinstance(obs, dict):
#         inp = obs.get("input") or {}
#     else:
#         inp = {}
    
#     if isinstance(inp, str):
#         try:
#             inp = json.loads(inp)
#         except Exception:
#             inp = {}
    
#     if isinstance(inp, dict):
#         ctx = inp.get("context_text") or (inp.get("kwargs") or {}).get("context_text")
#         question = (inp.get("kwargs") or {}).get("question")
#     else:
#         ctx = None
#         question = None
    
#     return ctx, question

# # 转换时间范围
# start_ts_local = to_local_naive(start_ts)
# end_ts_local = to_local_naive(end_ts)

# # 打印时间范围信息
# if start_ts_local and end_ts_local:
#     print(f"使用时间范围: {start_ts_local} 到 {end_ts_local}")
# elif start_ts_local:
#     print(f"使用时间范围: {start_ts_local} 到 现在")
# elif end_ts_local:
#     print(f"使用时间范围: 最早 到 {end_ts_local}")

# print(f"\n开始从 Langfuse 搜索数据...")
# print(f"搜索关键词: {NAME_KEYWORD}")

# # 创建 Langfuse 客户端
# client = FernLangfuse(
#     base_url=HOST,
#     username=PK,
#     password=SK,
#     timeout=120,
# )

# # 使用 SDK 直接查询
# all_obs = []
# page = 1
# page_size = 10
# max_pages = 2000

# print(f"开始分页查询（每页 {page_size} 条）...")

# while page <= max_pages:
#     try:
#         print(f"  请求第 {page} 页...", end=" ", flush=True)
        
#         # 使用 SDK 获取数据
#         result = client.observations.get_many(
#             page=page,
#             limit=page_size,
#             name=NAME_KEYWORD,
#             from_start_time=start_ts_local if start_ts_local else None,
#             to_start_time=end_ts_local if end_ts_local else None,
#         )
        
#         # 获取数据列表
#         items = result.data if hasattr(result, "data") else []
        
#         if not items:
#             print("无更多数据")
#             break
        
#         # 客户端过滤：确保时间在范围内（SDK 可能已经做了部分过滤）
#         filtered_items = []
#         for obs in items:
#             # 获取时间
#             obs_ts = None
#             if hasattr(obs, 'start_time') and obs.start_time:
#                 obs_ts = parse_ts(obs.start_time)
#             elif hasattr(obs, 'created_at') and obs.created_at:
#                 obs_ts = parse_ts(obs.created_at)
#             elif isinstance(obs, dict):
#                 obs_ts = parse_ts(
#                     obs.get("startTime")
#                     or obs.get("start_time")
#                     or obs.get("createdAt")
#                     or obs.get("created_at")
#                     or obs.get("timestamp")
#                 )
            
#             if obs_ts:
#                 obs_ts_local = to_local_naive(obs_ts)
#                 # 如果记录时间晚于结束时间，丢弃
#                 if end_ts_local and obs_ts_local and obs_ts_local > end_ts_local:
#                     continue
#                 # 如果记录时间早于开始时间，说明已经翻到窗口之前的数据，停止
#                 if start_ts_local and obs_ts_local and obs_ts_local < start_ts_local:
#                     print(f"\n检测到数据超出时间范围（{obs_ts_local} < {start_ts_local}），停止获取")
#                     page = max_pages + 1  # 退出循环
#                     break
            
#             filtered_items.append(obs)
        
#         if filtered_items:
#             all_obs.extend(filtered_items)
#             print(f"获取到 {len(items)} 条记录，过滤后 {len(filtered_items)} 条满足条件，累计 {len(all_obs)} 条")
#         else:
#             print(f"获取到 {len(items)} 条记录，但都不满足时间范围")
        
#         # 如果返回的记录数少于 page_size，说明已经是最后一页
#         if len(items) < page_size:
#             print("已获取所有数据")
#             break
        
#         page += 1
        
#     except Exception as e:
#         print(f"\n请求出错: {type(e).__name__}: {e}")
#         import traceback
#         print(f"错误详情: {traceback.format_exc()[:500]}")
#         break

# if len(all_obs) > 0:
#     print(f"\n成功获取到 {len(all_obs)} 条记录")
# else:
#     print(f"\n警告: 未获取到任何数据")

# # 提取上下文
# print(f"\n开始提取 {len(all_obs)} 条 observation 的上下文...")

# filtered = []
# for obs in all_obs:
#     # 获取基本信息
#     if hasattr(obs, 'name'):
#         name = obs.name
#     elif isinstance(obs, dict):
#         name = obs.get("name", "")
#     else:
#         name = ""
    
#     # 获取 trace 信息
#     trace = {}
#     if hasattr(obs, 'trace'):
#         trace_obj = obs.trace
#         if trace_obj:
#             if isinstance(trace_obj, dict):
#                 trace = trace_obj
#             else:
#                 trace = {
#                     "id": getattr(trace_obj, 'id', None),
#                     "name": getattr(trace_obj, 'name', None),
#                     "metadata": getattr(trace_obj, 'metadata', None),
#                 }
#     elif isinstance(obs, dict):
#         trace = obs.get("trace") or {}
    
#     trace_id = trace.get("id") if isinstance(trace, dict) else getattr(trace, 'id', None)
#     trace_name = trace.get("name") if isinstance(trace, dict) else getattr(trace, 'name', None)
#     metadata = trace.get("metadata") if isinstance(trace, dict) else getattr(trace, 'metadata', None)
    
#     # 提取时间戳
#     obs_ts = None
#     if hasattr(obs, 'start_time') and obs.start_time:
#         obs_ts = parse_ts(obs.start_time)
#     elif hasattr(obs, 'created_at') and obs.created_at:
#         obs_ts = parse_ts(obs.created_at)
#     elif isinstance(obs, dict):
#         obs_ts = parse_ts(
#             obs.get("startTime")
#             or obs.get("start_time")
#             or obs.get("createdAt")
#             or obs.get("created_at")
#             or obs.get("timestamp")
#         )
    
#     # 提取上下文和问题
#     ctx, q = extract_context(obs)
    
#     # 获取 input 和 output
#     if hasattr(obs, 'input'):
#         input_data = obs.input
#     elif isinstance(obs, dict):
#         input_data = obs.get("input")
#     else:
#         input_data = None
    
#     if hasattr(obs, 'output'):
#         output_data = obs.output
#     elif isinstance(obs, dict):
#         output_data = obs.get("output")
#     else:
#         output_data = None
    
#     # 获取 observation ID
#     if hasattr(obs, 'id'):
#         obs_id = obs.id
#     elif isinstance(obs, dict):
#         obs_id = obs.get("id")
#     else:
#         obs_id = None
    
#     filtered.append({
#         "trace_id": trace_id,
#         "observation_id": obs_id,
#         "name": name,
#         "trace_name": trace_name,
#         "metadata": metadata,
#         "question": q,
#         "context_text": ctx,
#         "input": input_data,
#         "output": output_data,
#         "obs_time": obs_ts.isoformat() if obs_ts else None,
#     })

# print(f"\n提取统计：")
# print(f"  总数: {len(all_obs)}")
# print(f"  成功提取: {len(filtered)}")

# # 保存结果
# out_dir = Path("../storage/testlog")
# out_dir.mkdir(parents=True, exist_ok=True)
# json_path = out_dir / f"{TEST_VER}_langfuse_contexts.json"
# json_path.write_text(json.dumps(filtered, ensure_ascii=False, indent=2), encoding="utf-8")

# print(f"\n提取到 {len(filtered)} 条满足条件的 observation")
# print(f"JSON 已保存: {json_path}")

# if filtered:
#     print("\n前3行示例：")
#     display_df = pd.DataFrame(filtered)
#     key_cols = ['name', 'question', 'context_text']
#     available_cols = [col for col in key_cols if col in display_df.columns]
#     if available_cols:
#         print(display_df[available_cols].head(3).to_string())
#     else:
#         print(display_df.head(3))
# else:
#     print("无记录，未生成示例表")


In [84]:
# 从 {TEST_VER}content.log 中直接提取上下文（新方法）

class ContextExtractor:
    """从 content.log 文件中提取上下文"""
    def __init__(self, content_log_path: Path):
        self.content_log_path = content_log_path
    
    def extract_contexts(self) -> List[Dict]:
        """从日志文件中提取每个问题的上下文"""
        contexts = []
        
        print(f"正在读取文件: {self.content_log_path}")
        
        with open(self.content_log_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        i = 0
        while i < len(lines):
            line = lines[i]
            
            # 查找包含 "问题:" 的行
            if '问题:' in line:
                # 提取问题文本（从 "问题: " 之后到行尾）
                question_match = re.search(r'问题:\s*(.+)', line)
                if question_match:
                    question_text = question_match.group(1).strip()
                    
                    # 查找后续的 "上下文文本:"
                    context_start_idx = None
                    for j in range(i + 1, min(i + 10, len(lines))):  # 在接下来的10行内查找
                        if '上下文文本:' in lines[j]:
                            context_start_idx = j + 1
                            break
                    
                    if context_start_idx:
                        # 提取上下文文本（直到下一个包含 "问题:" 的行）
                        context_lines = []
                        for j in range(context_start_idx, len(lines)):
                            next_line = lines[j]
                            # 如果遇到下一个包含 "问题:" 的行，停止
                            if '问题:' in next_line and j > context_start_idx:
                                break
                            context_lines.append(next_line)
                        
                        # 合并上下文文本
                        context_text = ''.join(context_lines).strip()
                        
                        if question_text and context_text:
                            contexts.append({
                                'question_text': question_text,
                                'context_text': context_text
                            })
                            
                            # 更新索引到下一个问题之前
                            i = j - 1  # j 是下一个问题的位置，减1因为循环会加1
                            continue
            
            i += 1
        
        print(f"成功提取 {len(contexts)} 个问题的上下文")
        return contexts

# 提取上下文
content_log_path = Path(f"../storage/testlog/{TEST_VER}content.log")
context_extractor = ContextExtractor(content_log_path)
contexts_list = context_extractor.extract_contexts()

print(f"\n=== 上下文提取完成 ===")
print(f"共提取 {len(contexts_list)} 个问题的上下文")
print()

# 显示前3个示例
if len(contexts_list) > 0:
    print("=== 前3个示例 ===")
    for idx, ctx in enumerate(contexts_list[:3], 1):
        print(f"\n示例 {idx}:")
        print(f"问题: {ctx['question_text'][:100]}...")
        print(f"上下文长度: {len(ctx['context_text'])} 字符")
        print(f"上下文预览: {ctx['context_text'][:200]}...")
else:
    print("未提取到任何上下文")


正在读取文件: ..\storage\testlog\0202content.log
成功提取 49 个问题的上下文

=== 上下文提取完成 ===
共提取 49 个问题的上下文

=== 前3个示例 ===

示例 1:
问题: 需要填写该公司所属的行业分类，应根据其主营业务范围，按照国家或行业标准的层级分类体系（如国民经济行业分类）准确填写，通常需参考营业执照、经营资质或企业公开信息中的行业归属信息。...
上下文长度: 10502 字符
上下文预览: <doc id=doc_1>
（一）行业属性与定位
</doc>

<doc id=doc_2>
（一）报告期内公司所属行业发展情况
中国铁建是全球最具实力、规模的特大型综合建设集团之一，业务涵盖工程承包、规划设计咨询、投资运营、房地产开发、工业制造、物资物流、绿色环保、产业金融及战略性新兴产业。
</doc>

<doc id=doc_3>
1. 工程承包
2025年上半年，依托“两重”项目与新...

示例 2:
问题: 需要填写负责本次授信调查的调查人员姓名，通常为两名或以上，信息应来源于内部调查任务分配记录或系统登记的调查人员名单。...
上下文长度: 16708 字符
上下文预览: <doc id=doc_1>
一、企业背景
</doc>

<doc id=doc_2>
| 序号 | 姓名 | 职务 |
| --- | --- | --- |
| 1 | 戴和根 | 董事长 |
| 2 | 朱宏标 | 财务负责人 |
| 3 | 郜烈阳 | 董事 |
| 4 | 解国光 | 董事 |
| 5 | 马传景 | 董事 |
| 6 | 倪真 | 董事 |
| 7 | 钱伟伦 | 董...

示例 3:
问题: 需要填写提交该授信初审报告的呈报单位名称，通常为分行或分支机构的全称，应根据实际负责该业务的机构信息进行填写。...
上下文长度: 25727 字符
上下文预览: <doc id=doc_1>
1. 法人 √适用□不适用
| 名称 | 中国铁道建筑集团有限公司 |
| --- | --- |
| 单位负责人或法定代表人 | 汪建平 |
| 成立日期 | 1990年8月28日 |
| 主要经营业务 | 铁路、地铁、公路、机场、港口、码头、隧道、桥梁、水利电力、邮电、矿山、林木、市政工程的技术咨询和线路、

In [85]:
# 匹配上下文和答案，合并并保存为 JSON

def match_and_merge(answers_df: pd.DataFrame, contexts_list: List[Dict]) -> List[Dict]:
    """匹配答案和上下文，合并为同一个 JSON"""
    merged_data = []
    
    # 创建上下文字典，以 question_text 为键
    context_dict = {}
    for ctx in contexts_list:
        question_text = ctx['question_text'].strip()
        context_dict[question_text] = ctx['context_text']
    
    print(f"答案数量: {len(answers_df)}")
    print(f"上下文数量: {len(contexts_list)}")
    print()
    
    # 匹配并合并
    matched_count = 0
    unmatched_count = 0
    
    for idx, row in answers_df.iterrows():
        question_id = row['question_id']
        question_text = row['question_text'].strip()
        question_answer = row['question_answer']
        answer_raw = row['answer_raw']
        
        # 尝试精确匹配
        context_text = context_dict.get(question_text)
        
        # 如果精确匹配失败，尝试模糊匹配（去除空格、标点等）
        if not context_text:
            # 标准化问题文本（去除多余空格）
            normalized_qtext = ' '.join(question_text.split())
            for ctx_qtext, ctx_text in context_dict.items():
                normalized_ctx_qtext = ' '.join(ctx_qtext.split())
                if normalized_qtext == normalized_ctx_qtext:
                    context_text = ctx_text
                    break
        
        # 如果还是没找到，尝试部分匹配（问题文本的前80%匹配）
        if not context_text:
            qtext_prefix = normalized_qtext[:int(len(normalized_qtext) * 0.8)] if normalized_qtext else ""
            for ctx_qtext, ctx_text in context_dict.items():
                normalized_ctx_qtext = ' '.join(ctx_qtext.split())
                if normalized_ctx_qtext.startswith(qtext_prefix) and len(qtext_prefix) > 20:
                    context_text = ctx_text
                    break
        
        merged_item = {
            'question_id': question_id,
            'question_text': question_text,
            'question_answer': question_answer,
            'answer_raw': answer_raw,
            'context_text': context_text if context_text else None
        }
        
        merged_data.append(merged_item)
        
        if context_text:
            matched_count += 1
        else:
            unmatched_count += 1
    
    print(f"匹配统计:")
    print(f"  成功匹配: {matched_count} 个")
    print(f"  未匹配: {unmatched_count} 个")
    print()
    
    return merged_data

# 检查 answers_df 是否存在
if 'answers_df' in globals() and len(answers_df) > 0:
    # 匹配并合并
    merged_list = match_and_merge(answers_df, contexts_list)
    
    # 保存为 JSON
    output_dir = Path("../storage/testlog")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / f"{TEST_VER}test_answers_with_context.json"
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(merged_list, f, ensure_ascii=False, indent=2)
    
    print(f"=== 合并结果已保存 ===")
    print(f"保存路径: {output_file}")
    print(f"共保存 {len(merged_list)} 条记录")
    print()
    
    # 显示前3个示例
    print("=== 前3个合并结果示例 ===")
    for idx, item in enumerate(merged_list[:3], 1):
        print(f"\n示例 {idx}:")
        print(f"问题ID: {item['question_id']}")
        print(f"问题文本: {item['question_text'][:80]}...")
        print(f"是否有上下文: {'是' if item['context_text'] else '否'}")
        if item['context_text']:
            print(f"上下文长度: {len(item['context_text'])} 字符")
            print(f"上下文预览: {item['context_text'][:150]}...")
        print(f"答案预览: {item['answer_raw'][:100]}...")
else:
    print("错误: 未找到 answers_df，请先运行 '6. 问题答案提取模块'")


答案数量: 75
上下文数量: 49

匹配统计:
  成功匹配: 49 个
  未匹配: 26 个

=== 合并结果已保存 ===
保存路径: ..\storage\testlog\0202test_answers_with_context.json
共保存 75 条记录

=== 前3个合并结果示例 ===

示例 1:
问题ID: q_1
问题文本: 需要填写提交该授信初审报告的呈报单位名称，通常为分行或分支机构的全称，应根据实际负责该业务的机构信息进行填写。...
是否有上下文: 是
上下文长度: 25727 字符
上下文预览: <doc id=doc_1>
1. 法人 √适用□不适用
| 名称 | 中国铁道建筑集团有限公司 |
| --- | --- |
| 单位负责人或法定代表人 | 汪建平 |
| 成立日期 | 1990年8月28日 |
| 主要经营业务 | 铁路、地铁、公路、机场、港口、码头、隧道、桥梁、水利电力、邮...
答案预览: 提交该授信初审报告的呈报单位名称应为北京银行脱敏B分行，因为该报告由北京银行脱敏B分行开展尽职调查并作为信贷审批的基础资料，且报告中明确提及调查机构为‘北京银行脱敏B分行’，调查结论亦由该分行出具并纳...

示例 2:
问题ID: q_10
问题文本: 判断该公司是否属于政府融资平台（包括监管类或监测类），需根据企业股权结构、实际控制人、融资用途及政府相关文件等信息进行核实。...
是否有上下文: 是
上下文长度: 42200 字符
上下文预览: <doc id=doc_1>
七、企业发展
七、企业发展
当前表格的名字以及单位信息: 7.1 融资信息
| 序号 | 发布日期 | 融资轮次 | 融资金额 | 投资方 |
| --- | --- | --- | --- | --- |
| 1 | 2022-12-19 | 战略投资 | 100亿元...
答案预览: 该公司不属于政府融资平台。根据股权结构，中国铁建股份有限公司的控股股东为中国铁道建筑集团有限公司，而中国铁道建筑集团有限公司的实际控制人为国务院国有资产监督管理委员会（国资委），表明该公司为中央企业，...

示例 3:
问题ID: q_11
问题文本: 判断该公司是否为银行的关联方，需根据企业股权结构、管理层关联关系、实际控制人信息等进行核实，以确定是否存